In [ ]:
import os
import numpy as np
from ase.io import read
from ase.io import write
from ase.optimize import BFGS
from xtb.ase.calculator import XTB   # pip install ase xtb-python (recommended)
# If no xTB: you can skip pre-opt or use PySCF later

from rdkit import Chem
from rdkit.Chem import AllChem

from pyscf import gto, scf
from pyscf.lib import logger
from asf.wrapper import find_from_mol, find_from_scf
from asf.utility import pictures_Jmol   # optional for visualization

In [ ]:


print("=== Full Workflow: 1CA2 → Ligand Placement → ASF → DMET (PennyLane) ===")

# ===================================================================
# 1. Extract Zn-centered cluster from 1CA2.pdb
# ===================================================================
atoms = read("/home/loharkar/QuEnAIS-quantum-embedding/data/raw/1CA2.pdb")

# Find Zn (first occurrence)
zn_idx = [i for i, a in enumerate(atoms) if a.symbol == "Zn"][0]
zn_pos = atoms[zn_idx].position.copy()

# Select atoms within ~6.5 Å (better than 5 Å for testing)
sel = [i for i, a in enumerate(atoms) 
       if np.linalg.norm(a.position - zn_pos) < 6.5]

cluster = atoms[sel]
write("cluster.xyz", cluster)
print(f"Cluster extracted: {len(cluster)} atoms, centered on Zn")

# ===================================================================
# 2. Prepare small ligands with RDKit
# ===================================================================
smiles = {
    "L1": "c1ncc[nH]1",      # imidazole
    "L2": "Cc1ncc[nH]1",     # 4-methylimidazole
    "L3": "c1ccncc1"         # pyridine
}

for name, smi in smiles.items():
    mol = Chem.MolFromSmiles(smi)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.UFFOptimizeMolecule(mol)
    Chem.MolToXYZFile(mol, f"{name}.xyz")
    print(f"{name} ligand generated")

# ===================================================================
# 3. Realistic ligand placement + cheap pre-optimization (for each ligand)
# ===================================================================
target_distance = 2.00   # Zn–N distance (standard for CA inhibitors)

for name in smiles.keys():
    cluster = read("cluster.xyz")
    lig = read(f"{name}.xyz")
    
    # Find Zn and coordinating N (first N atom)
    zn_idx = [i for i, a in enumerate(cluster) if a.symbol == "Zn"][0]
    zn_pos = cluster[zn_idx].position.copy()
    
    n_idx = [i for i, a in enumerate(lig) if a.symbol == "N"][0]
    
    # Place ligand N at 2.0 Å along z (good enough for testing; improve later with water vector)
    lig.positions -= lig.positions[n_idx]
    direction = np.array([0.0, 0.0, 1.0])
    lig.positions += zn_pos + target_distance * direction
    
    combined = cluster + lig
    write(f"{name}_complex_initial.xyz", combined)
    
    # --- Cheap pre-optimization (relax ligand only) ---
    # Tag protein atoms as frozen (tag=1)
    for i in range(len(cluster)):
        combined[i].tag = 1
    
    calc = XTB(method="GFN2-xTB")
    combined.calc = calc
    
    opt = BFGS(combined, trajectory=f"{name}_opt.traj", logfile=f"{name}_opt.log")
    opt.run(fmax=0.05)   # loose convergence is fine for testing
    
    write(f"{name}_complex_opt.xyz", combined)
    print(f"{name} complex optimized and saved")

# ===================================================================
# 4. Run ASF (Active Space Finder) on the optimized complex
# ===================================================================
# Choose one ligand for now (repeat for L2, L3)
ligand_name = "L1"   # change to L2 or L3

mol = gto.Mole()
mol.atom = f"{ligand_name}_complex_opt.xyz"
mol.basis = 'def2-SVP'          # recommended in ASF examples
mol.charge = 0                  # Try 0 first. If SCF fails badly → try 2 (Zn²⁺ model)
mol.spin = 0
mol.verbose = logger.INFO
mol.build()

print(f"\nRunning ASF on {ligand_name} complex (charge={mol.charge}, spin={mol.spin})")

nel, mo_list, mo_coeff = runasf_from_mole(
    mol,
    entropy_threshold=0.15,     # slightly lower than default for metal systems
    states=1,                   # ground state singlet
    sort_mos=True
)

print(f"\n=== ASF RESULT for {ligand_name} ===")
print(f"Active electrons : {nel}")
print(f"Active orbitals  : {len(mo_list)}")
print(f"Orbital indices  : {mo_list}")

# Optional: visualize selected active orbitals
pictures_Jmol(mol, mo_coeff, mo_list=mo_list, rotate=(-45, 0, 0))

# ===================================================================
# 5. DMET setup using ASF active orbitals to define impurity (minimal manual part)
# ===================================================================
# Note: Full self-consistent DMET with libdmet on molecular systems is a bit more involved.
# For initial testing we extract the impurity Hamiltonian after one mean-field + ASF guidance.

# Run a good mean-field (better than UHF)
mf = scf.RKS(mol)          # or RHF
mf.xc = 'b3lyp'
mf.kernel()

# Simple way to define impurity: use ASF active orbitals as starting point for impurity
imp_idx = list(mo_list)    # ASF-selected orbitals go into the impurity
# You can expand this with all orbitals on Zn + ligand atoms if you want a larger impurity

print(f"Impurity defined with {len(imp_idx)} orbitals from ASF")

# ===================================================================
# 6. Convert impurity Hamiltonian to PennyLane qubit operator
# ===================================================================
# For a quick start we can build the full Hamiltonian and project, but the proper way 
# follows the PennyLane DMET tutorial you shared.

# Placeholder: full molecular Hamiltonian (for small testing only)
from pyscf import ao2mo
from pennylane.qchem import one_particle, two_particle, observable
import pennylane as qml

# Example: build full Hamiltonian (use only for very small clusters)
h1 = mf.get_hcore()
h2 = ao2mo.restore(1, ao2mo.kernel(mol, mf.mo_coeff), mol.nao_nr())

one_elec = one_particle(h1)
two_elec = two_particle(h2)
qubit_op = observable([one_elec, two_elec], mapping="jordan_wigner")

print("\n=== Qubit Hamiltonian ready for quantum solver ===")
print(qubit_op)

# You can now use this with VQE, QPE, or any PennyLane solver:
# dev = qml.device("default.qubit", wires=qubit_op.wires)
# @qml.qnode(dev)
# def circuit(params):
#     ... your ansatz ...
#     return qml.expval(qubit_op)

print("\nWorkflow finished for", ligand_name)
print("Next steps:")
print(" - Repeat for L2 and L3")
print(" - Tune charge/spin if SCF/ASF fails")
print(" - Improve impurity definition using atom indices + ASF orbitals")
print(" - Implement full self-consistent DMET loop from the PennyLane tutorial")

In [ ]:
import numpy as np
from ase.io import read, write
from ase.optimize import BFGS
from xtb.ase.calculator import XTB   # recommended for pre-opt

from rdkit import Chem
from rdkit.Chem import AllChem

from pyscf import gto, scf
from pyscf.lib import logger

# === Correct modern ASF imports ===
from asf.wrapper import find_from_mol, find_from_scf
from asf.utility import pictures_Jmol

print("=== Full Workflow: 1CA2.pdb → Realistic Ligand Placement → ASF → PennyLane ===")

# ===================================================================
# 1. Extract Zn-centered cluster from 1CA2.pdb (6.5 Å is good for testing)
# ===================================================================
atoms = read("/home/loharkar/QuEnAIS-quantum-embedding/data/raw/1CA2.pdb")

zn_idx = [i for i, a in enumerate(atoms) if a.symbol == "Zn"][0]
zn_pos = atoms[zn_idx].position.copy()

sel = [i for i, a in enumerate(atoms) 
       if np.linalg.norm(a.position - zn_pos) < 4.0]

cluster = atoms[sel]
write("cluster.xyz", cluster)
print(f"Cluster extracted: {len(cluster)} atoms")
print("STEP 1 DONE")
# ===================================================================
# 2. Generate ligands (L1, L2, L3)
# ===================================================================
smiles = {
    "L1": "c1ncc[nH]1",      # imidazole
    "L2": "Cc1ncc[nH]1",
    "L3": "c1ccnc
}

for name, smi in smiles.items():
    mol_rd = Chem.MolFromSmiles(smi)
    mol_rd = Chem.AddHs(mol_rd)
    AllChem.EmbedMolecule(mol_rd, randomSeed=42)
    AllChem.UFFOptimizeMolecule(mol_rd)
    Chem.MolToXYZFile(mol_rd, f"{name}.xyz")

print("STEP 2 DONE")
# ===================================================================
# 3. Ligand placement at Zn–N = 2.0 Å  (NO pre-optimization for now)
# ===================================================================
ligand_name = "L1"   # change to L2 or L3 when ready

cluster = read("cluster.xyz")
lig = read(f"{ligand_name}.xyz")

zn_idx = [i for i, a in enumerate(cluster) if a.symbol == "Zn"][0]
zn_pos = cluster[zn_idx].position.copy()

n_idx = [i for i, a in enumerate(lig) if a.symbol == "N"][0]

# Place coordinating nitrogen at 2.0 Å
lig.positions -= lig.positions[n_idx]
lig.positions += zn_pos + np.array([0.0, 0.0, 2.0])

combined = cluster + lig

# Ensure no PBC and center the system
combined.set_pbc(False)
combined.set_cell([0.0, 0.0, 0.0])
combined.center()

write(f"{ligand_name}_complex_initial.xyz", combined)
print(f"{ligand_name} complex placed at Zn–N = 2.0 Å (no pre-opt)")

print("STEP 3 DONE")
# ===================================================================
# 4. ASF — Active Space Selection (modern API)
# ===================================================================

xyz_filename = f"{ligand_name}_complex_initial.xyz"
mol = gto.Mole()

mol.atom = xyz_filename
mol.basis = 'def2-SVP'
mol.charge = 2          # Important: try 0 first. If unstable, change to 2
mol.spin = 0
mol.verbose = logger.INFO
mol.build()


print(f"Molecule built: {mol.nao_nr()} basis functions, charge={mol.charge}")

mf = scf.UKS(mol)
mf.xc = 'b3lyp'
mf.conv_tol = 1e-6
mf.conv_tol_grad = 1e-3      # looser gradient tolerance for difficult cases
mf.max_cycle = 300

# Strong convergence aids for metal clusters
mf.damp = 0.8                # heavy damping
mf.level_shift = 0.4         # strong level shift
mf.diis_space = 20
mf.diis_start_cycle = 8
mf.direct_scf_tol = 1e-13

# Optional: try a different functional if this still struggles
# mf.xc = 'wb97xd'

print("Starting robust UKS calculation...")
mf.kernel()

print(f"SCF converged: {mf.converged}")
print(f"Final energy: {mf.e_tot}")




# Now use find_from_scf (more control)
active_space = find_from_scf(
    mf,
    entropy_threshold=0.15,      # tune: 0.1–0.25 works well for metal-ligand systems
    states=1,                    # ground state
    sort_mos=True                # reorder MOs for direct use in CASSCF/CASCI
)

print(f"\n=== ASF RESULT for {ligand_name} ===")
print(f"Active electrons : {active_space.nel}")
print(f"Active orbitals  : {active_space.norb}")
print(f"Active orbital indices : {active_space.mo_list}")

# Optional visualization
# pictures_Jmol(mol, active_space.mo_coeff, mo_list=active_space.mo_list)

# ===================================================================
# 5. DMET: Use ASF active orbitals to define the impurity region
# ===================================================================
imp_idx = list(active_space.mo_list)   # ASF gives us the strongly correlated orbitals

print(f"\nImpurity region defined using {len(imp_idx)} ASF-selected orbitals "
      f"(Zn-ligand correlation captured)")

# ===================================================================
# 6. Build qubit Hamiltonian for PennyLane (simplified version for testing)
# ===================================================================
from pyscf import ao2mo
from pennylane.qchem import one_particle, two_particle, observable

h1 = mf.get_hcore()
h2 = ao2mo.restore(1, ao2mo.kernel(mol, mf.mo_coeff), mol.nao_nr())

one_elec = one_particle(h1)
two_elec = two_particle(h2)
qubit_ham = observable([one_elec, two_elec], mapping="jordan_wigner")

print("\n=== Qubit Hamiltonian ready for quantum solver (VQE, QPE, etc.) ===")
print(qubit_ham)

print("\nWorkflow completed for", ligand_name)
print("Tips:")
print(" - Repeat for L2 and L3 by changing ligand_name")
print(" - If SCF is unstable, set mol.charge = 2 and/or use UKS")
print(" - For production: implement the full self-consistent DMET loop from the PennyLane tutorial")

=== Full Workflow: 1CA2.pdb → Realistic Ligand Placement → ASF → PennyLane ===
Cluster extracted: 15 atoms
STEP 1 DONE
STEP 2 DONE
L1 complex placed at Zn–N = 2.0 Å (no pre-opt)
STEP 3 DONE
System: uname_result(system='Linux', node='erebos02', release='5.15.0-170-generic', version='#180-Ubuntu SMP Fri Jan 9 16:10:31 UTC 2026', machine='x86_64')  Threads 24
Python 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:45:40) [GCC 14.3.0]
numpy 2.4.3  scipy 1.17.1  h5py 3.16.0
Date: Tue Apr  7 12:00:26 2026
PySCF version 2.12.1
PySCF path  /home/loharkar/QuEnAIS-quantum-embedding/quenais-env2/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 24
[INPUT] num. electrons = 158
[INPUT] charge = 2
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[I

In [ ]:


# ===================================================================
# 3. Ligand placement + pre-optimization (fixed version)
# ===================================================================
ligand_name = "L1"   # change as needed

cluster = read("cluster.xyz")
lig = read(f"{ligand_name}.xyz")

zn_idx = [i for i, a in enumerate(cluster) if a.symbol == "Zn"][0]
zn_pos = cluster[zn_idx].position.copy()

n_idx = [i for i, a in enumerate(lig) if a.symbol == "N"][0]

# Place ligand
lig.positions -= lig.positions[n_idx]
lig.positions += zn_pos + np.array([0.0, 0.0, 2.0])

combined = cluster + lig

# === CRITICAL FIX: Disable periodic boundary conditions ===
combined.set_pbc(False)        # This is the most important line
# Optional: also clear any cell if present
combined.set_cell([0, 0, 0])

write(f"{ligand_name}_complex_initial.xyz", combined)

# Pre-optimize: freeze the protein cluster, relax only the ligand + Zn
for i in range(len(cluster)):
    combined[i].tag = 1   # tag=1 means frozen for many optimizers/calculators

calc = XTB(method="GFN2-xTB")
combined.calc = calc

opt = BFGS(combined, logfile=f"{ligand_name}_opt.log")
opt.run(fmax=0.05)   # loose tolerance is sufficient for pre-opt

write(f"{ligand_name}_complex_opt.xyz", combined)
print(f"{ligand_name} complex prepared and pre-optimized (PBC disabled)")